# Phi-3 Fine-Tuning for Industrial Maintenance

This notebook fine-tunes Microsoft's Phi-3 Mini (3.8B parameters) on steel plant maintenance data using QLoRA and Unsloth optimization.

**Training Configuration:**
- Base Model: `unsloth/Phi-3-mini-4k-instruct`
- Method: 4-bit quantization with LoRA adapters (rank 16)
- Hardware: Google Colab T4 GPU (free tier)
- Dataset: 1,776 training samples, 197 validation samples
- Training Duration: ~4-5 hours

**Prerequisites:**
- Google Colab with T4 GPU runtime
- Training data stored in Google Drive at `Alloy-Agent/data/training/`
- Weights & Biases account for experiment tracking

---

## Environment Setup

Install required packages and resolve CUDA compatibility issues for bitsandbytes.

In [ ]:
# Install core dependencies
!pip install -q -U "torchvision>=0.27.0"
!pip install -q -U "bitsandbytes>=0.48.0"
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "transformers>=4.51.3,<=5.5.0" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0"
!pip install -q -U accelerate peft wandb

# Fix CUDA library mismatch (Colab uses CUDA 13, but leaves CUDA 12 artifacts)
!pip uninstall -y nvidia-nvjitlink-cu12
!pip install nvidia-nvjitlink

import os
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-13.0/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

## Data Access

Mount Google Drive and verify training data is accessible.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define data paths
TRAIN_PATH = '/content/drive/MyDrive/Alloy-Agent/data/training/train.jsonl'
VAL_PATH = '/content/drive/MyDrive/Alloy-Agent/data/training/val.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/Alloy-Agent/models/fine_tuned_phi3'

# Verify data exists
assert os.path.exists(TRAIN_PATH), f"Training data not found at {TRAIN_PATH}"
assert os.path.exists(VAL_PATH), f"Validation data not found at {VAL_PATH}"

print("Data verified")

## Experiment Tracking

Configure Weights & Biases for monitoring training metrics.

In [ ]:
import wandb
wandb.login()

## Model Initialization

Load Phi-3 Mini with 4-bit quantization and configure LoRA adapters for parameter-efficient fine-tuning.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

# Load base model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect optimal dtype
    load_in_4bit=True,
)

# Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,  # Regularization
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## Dataset Preparation

Load and tokenize the maintenance dataset. Data format:
```json
{
  "instruction": "Analyze equipment degradation...",
  "input": "Equipment: Air Compressor | Sensor readings...",
  "output": "DIAGNOSIS: ... ROOT CAUSE: ... RECOMMENDATIONS: ..."
}
```

In [ ]:
from datasets import load_dataset

# Load datasets
train_dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')
val_dataset = load_dataset('json', data_files=VAL_PATH, split='train')

print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")

In [ ]:
def format_chat_template(instruction, input_text, output_text):
    """Format data as Phi-3 chat messages."""
    return f"""<|system|>You are an industrial maintenance AI assistant specialized in steel plant equipment analysis.<|end|>
<|user|>{instruction}\n\n{input_text}<|end|>
<|assistant|>{output_text}<|end|>"""

def tokenize_function(examples):
    """Tokenize batched examples with chat formatting."""
    formatted_texts = [
        format_chat_template(inst, inp, out)
        for inst, inp, out in zip(examples['instruction'], examples['input'], examples['output'])
    ]
    
    outputs = tokenizer(
        formatted_texts,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

# Tokenize datasets
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

print("Tokenization complete")

## Training Configuration

Configure hyperparameters and training strategy.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling, EarlyStoppingCallback

training_args = TrainingArguments(
    # Output configuration
    output_dir="./alloy-phi3-output",
    
    # Training schedule
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 8
    
    # Optimizer settings
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    max_grad_norm=1.0,
    optim="adamw_8bit",
    
    # Precision
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # Logging and checkpointing
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,  # Keep only 3 latest checkpoints
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    
    # Experiment tracking
    report_to="wandb",
    run_name="phi3-maintenance-training",
    
    # Memory optimization
    gradient_checkpointing=True,
    seed=42,
)

# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

## Training Execution

Start fine-tuning. Expected duration: 4-5 hours on T4 GPU.

In [ ]:
# Initialize WandB run
wandb.init(
    project="alloy-agent-phi3",
    name="phi3-maintenance-training",
    config={
        "model": "Phi-3-mini-4k-instruct",
        "method": "QLoRA",
        "lora_r": 16,
        "batch_size": 8,
        "learning_rate": 2e-4,
        "epochs": 3,
    }
)

# Start training
trainer.train()

print("Training complete")

## Model Export

Save the fine-tuned LoRA adapters to Google Drive.

In [ ]:
# Save model and tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

wandb.finish()

print(f"Model saved to: {OUTPUT_DIR}")

## Inference Testing

Validate the fine-tuned model with a sample query.

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test prompt
test_prompt = """<|system|>You are an industrial maintenance AI assistant specialized in steel plant equipment analysis.<|end|>
<|user|>Analyze this sensor data: temperature=85°C, vibration=0.8mm/s, pressure=2.1bar. What's the maintenance recommendation?<|end|>
<|assistant|>"""

# Generate response
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)

print("\nSample inference:")
print(response)

## Deployment Guide

The fine-tuned model can be deployed using:

### Option 1: HuggingFace Inference API (Recommended)
```python
# Upload model to HuggingFace Hub
from huggingface_hub import login
login()

model.push_to_hub("your-username/alloy-agent-phi3")
tokenizer.push_to_hub("your-username/alloy-agent-phi3")

# Use Inference API
from huggingface_hub import InferenceClient
client = InferenceClient(model="your-username/alloy-agent-phi3")
response = client.text_generation(prompt, max_new_tokens=500)
```

### Option 2: Local Inference
Load the model from Google Drive in production:
```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="path/to/fine_tuned_phi3",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

---

**Training Metrics:**
- Total training steps: ~666
- Checkpoints saved every 100 steps
- Best model selected based on validation loss
- Early stopping with patience=3

**Model Stats:**
- Base parameters: 3.85B
- Trainable parameters: ~30M (0.78% of total)
- Quantization: 4-bit
- Memory footprint: ~3GB on GPU